In [1]:
import torch 
import torch.nn as nn

In [ ]:
import torch
import torch.nn as nn

class CAM(nn.Module):
    """
    Channel Attention Module (CAM)
    Focuses on 'what' is important across the channel dimension.
    """
    def __init__(self, in_channels, reduction=16):
        super(CAM, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        
        # Shared MLP implemented via 1x1 Convolutions to avoid flattening/unflattening
        self.shared_mlp = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction, kernel_size=1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_channels // reduction, in_channels, kernel_size=1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Apply shared MLP on both pooling outputs
        avg_out = self.shared_mlp(self.avg_pool(x))
        max_out = self.shared_mlp(self.max_pool(x))
        
        # Element-wise summation and activation
        attention = self.sigmoid(avg_out + max_out)
        
        # Scale the original input features
        return x * attention


class SAM(nn.Module):
    """
    Spatial Attention Module (SAM)
    Focuses on 'where' the important features are located spatially.
    """
    def __init__(self, kernel_size=7):
        super(SAM, self).__init__()
        # Padding is calculated to keep the spatial dimensions identical (e.g., (7-1)//2 = 3)
        padding = (kernel_size - 1) // 2
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Pool along the channel axis (dim=1)
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        
        # Concatenate spatial statistics
        concat = torch.cat([avg_out, max_out], dim=1)
        
        # 7x7 Convolution followed by Sigmoid
        attention = self.sigmoid(self.conv(concat))
        
        # Scale the feature map
        return x * attention


class CBAM(nn.Module):
    """
    Convolutional Block Attention Module (CBAM)
    Sequentially combines Channel and Spatial Attention.
    """
    def __init__(self, in_channels, reduction=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.cam = CAM(in_channels, reduction)
        self.sam = SAM(kernel_size)

    def forward(self, x):
        x = self.cam(x)
        x = self.sam(x)
        return x


class CNN(nn.Module):
    """
    A sample CNN architecture showing how to embed CBAM into feature blocks.
    """
    def __init__(self, num_classes=10):
        super(CNN, self).__init__()
        
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            CBAM(in_channels=64),  # Refines 64-channel map
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Block 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            CBAM(in_channels=128), # Refines 128-channel map
            nn.AdaptiveAvgPool2d((1, 1))
        )
        
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


# Verification check
if __name__ == "__main__":
    # Create a mock batch of 4 images with shape: (Batch Size, Channels, Height, Width)
    mock_input = torch.randn(4, 3, 32, 32)
    model = CNN(num_classes=10)
    output = model(mock_input)
    print(f"Input shape:  {mock_input.shape}")
    print(f"Output shape: {output.shape}")

Input to CBAM: torch.Size([4, 64, 32, 32])
Output from CBAM: torch.Size([4, 64, 32, 32])
Input to CBAM: torch.Size([4, 128, 16, 16])
Output from CBAM: torch.Size([4, 128, 16, 16])
Input shape:  torch.Size([4, 3, 32, 32])
Output shape: torch.Size([4, 10])
